# Crowd / 02 — Preprocessing

Deduplicate monthly observations, encode month cyclically and create a training-only weather snapshot.

Run every cell from top to bottom. Use the project `.venv` kernel in VS Code. No season field is used. Data and test scores are synthetic.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml_service/workflows.py").exists())
sys.path.insert(0, str(ROOT))
from ml_service import workflows as w
print("Project:", ROOT)
print("Synthetic demonstration: real-world accuracy has not been measured.")

Project: /workspace/scratch/71e3a62599e6/YatraSense_Organic
Synthetic demonstration: real-world accuracy has not been measured.


In [2]:
print(inspect.getsource(w.preprocess_crowd))
print(inspect.getsource(w.preprocessing))

def preprocess_crowd():
    df=pd.read_csv(DATA/'ingested.csv')
    # All customer-related columns are discarded; one example per place/month.
    fields=['destination_month_id','destination_id','destination_name','area','year','month','temperature_mean_c','precipitation_mm_month','wind_speed_kmh_mean','visitor_count','busyness_label','dataset_split']
    df=calendar_features(df[fields].drop_duplicates('destination_month_id'))
    assert len(df)==2400 and df.destination_month_id.is_unique
    assert df.groupby('dataset_split').year.agg(['min','max']).to_dict()=={'min':{'test':2024,'train':2016,'validation':2023},'max':{'test':2025,'train':2022,'validation':2023}}
    df.to_csv(DATA/'crowd.csv',index=False)
    train=df[df.dataset_split=='train']
    # This is a synthetic climatology for demo inference, never a live forecast.
    weather=train.groupby(['destination_id','month'])[['temperature_mean_c','precipitation_mm_month','wind_speed_kmh_mean']].mean().round(2).reset_index()
    weat

In [3]:
summary = w.preprocess_crowd()
print(summary)
df = pd.read_csv(w.DATA / "crowd.csv")
display(df.head())

{'test': 480, 'train': 1680, 'validation': 240}


,destination_month_id,destination_id,destination_name,area,year,month,temperature_mean_c,precipitation_mm_month,wind_speed_kmh_mean,visitor_count,busyness_label,dataset_split,month_sin,month_cos
0,YS001-2016-01,YS001,Pashupatinath,Kathmandu,2016,1,7.2,11.2,4.2,43412,Medium,train,0.5,0.866025
1,YS002-2016-01,YS002,Boudhanath,Kathmandu,2016,1,9.2,11.8,7.7,30616,Low,train,0.5,0.866025
2,YS003-2016-01,YS003,Swayambhunath,Kathmandu,2016,1,5.3,10.0,7.3,29015,Low,train,0.5,0.866025
3,YS004-2016-01,YS004,Kathmandu Durbar Square,Kathmandu,2016,1,9.1,11.2,6.0,39063,Medium,train,0.5,0.866025
4,YS005-2016-01,YS005,Garden of Dreams,Kathmandu,2016,1,6.7,13.1,8.5,7855,Low,train,0.5,0.866025


Numeric imputation and scaling, plus categorical imputation and one-hot encoding, are fitted inside each training pipeline. Validation and test records never fit those transforms. Recommendation crowd features are grouped out-of-fold predictions for training and frozen train-only predictions for later years. Those folds group whole years; they are cross-fitting, not a rolling historical backtest.